# Import the Libraries

In [ ]:
import pandas as pd
import numpy as np
import glob
import re
import os

# Import Folder Path

In [ ]:
folder_path = r"C:\Users\lenovo\Downloads\Survey of india Files Data"

# Import Csv Files 

# Fetch July Data

In [ ]:
import pandas as pd

# Read CSV
df = pd.read_csv(
    r"C:\Users\lenovo\Downloads\Survey of india Files Data\Region 1\region-1-1-2026-Jul-Aug.csv"
)

# Convert Date safely
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

# July 2026 date range
start_date = pd.Timestamp("2026-07-01")
end_date = pd.Timestamp("2026-07-31")

# Keep only July data
filtered_df = df.loc[
    df["Date"].between(start_date, end_date)
].copy()

print("Start Date:", start_date.strftime("%d-%m-%Y"))
print("End Date:", end_date.strftime("%d-%m-%Y"))
print("Total records:", len(filtered_df))
print(filtered_df.head())


# Drop Date Column

# Prepare July Data

In [ ]:
# Work on a separate copy so the original df remains unchanged.
combined_df = filtered_df.copy()

print("combined_df shape:", combined_df.shape)
print("Columns:", combined_df.columns.tolist())


# Date Already Available

In [ ]:
# Date already comes from the CSV and has been converted to datetime in Cell 6.
# Do NOT drop and reconstruct Date from FilePath.
combined_df["Date"] = pd.to_datetime(
    combined_df["Date"],
    errors="coerce"
)


In [ ]:
# Final July filter (kept here as a safety check)
combined_df = combined_df.loc[
    combined_df["Date"].between(start_date, end_date)
].copy()

print(combined_df["Date"].min(), "to", combined_df["Date"].max())
print("Rows:", len(combined_df))


In [ ]:
combined_df.head()


# Add New Column Fetch The Station Code

In [ ]:
combined_df["Station"] = (
    combined_df["FileName"]
    .astype(str)
    .str[:4]
    .str.upper()
)

combined_df["Extension"] = (
    combined_df["FileName"]
    .astype(str)
    .str.rsplit(".", n=1)
    .str[-1]
    .str.lower()
)

combined_df.head()


# Extension

In [ ]:
combined_df["Extension"] = combined_df["FileName"].apply(lambda x: os.path.basename(x[-3:]))
combined_df

# State Filter

In [ ]:
station_df = pd.read_excel(r"C:\Users\lenovo\Downloads\Region 2 Stations.xlsx")
station_df

# Mark Specific State

In [ ]:
specific_state = station_df[
    (station_df["State"] == "RAJASTHAN")  
]
specific_state

# Check the state Both File have or not

In [ ]:
# Keep only stations belonging to the selected state.
state_station_codes = set(
    specific_state["StationCodeRenamed"]
    .dropna()
    .astype(str)
    .str.upper()
)

state_df = combined_df.loc[
    combined_df["Station"].isin(state_station_codes)
].copy()

print("Selected state:", "RAJASTHAN")
print("Rows:", len(state_df))
print("Stations found:", state_df["Station"].nunique())
state_df.head()


In [ ]:
state_output = r"C:\Users\lenovo\Desktop\Survey Of India 2026 Files\July_Files\Rajasthan\Rajasthan's Meta Data for the period July 2026.xlsx"

state_df.to_excel(
    state_output,
    index=False
)

print("Saved:", state_output)


# State Comparison

In [ ]:
expected_station_codes = set(
    specific_state["StationCodeRenamed"]
    .dropna()
    .astype(str)
    .str.upper()
)

actual_station_codes = set(state_df["Station"].dropna().astype(str).str.upper())

missing_in_data = expected_station_codes - actual_station_codes
unexpected_in_data = actual_station_codes - expected_station_codes

if not missing_in_data and not unexpected_in_data:
    print("All station codes match.")
else:
    print("Missing station codes in data:", sorted(missing_in_data))
    print("Unexpected station codes in data:", sorted(unexpected_in_data))


# Checking Invalid Files


In [ ]:
import re

pattern2 = r'^[A-Za-z_]{4}\d{3}[A-Za-z]{2}\.zip$'
pattern3 = r'^[A-Za-z_]{3}\d{1}\d{3}[A-Za-z]{2}\.zip$'
pattern4 = r'^[A-Za-z_]{4}\d{3}[A-Za-z]\.T02$'
pattern5 = r'^[A-Za-z_]{3}\d{1}\d{3}[A-Za-z]\.T02$'
pattern6 = r'^[A-Za-z_]{4}\d{3}[A-Za-z]\d{2}[A-Za-z]\.zip$'

patterns = [pattern2, pattern3, pattern4, pattern5, pattern6]

filename_series = state_df["FileName"].astype(str)

valid_mask = filename_series.apply(
    lambda x: any(re.fullmatch(pattern, x) for pattern in patterns)
)

non_matching_files = state_df.loc[~valid_mask].copy()

print("Files with non-matching patterns:")
print(non_matching_files[["Date", "FileName", "Station"]].head(20))
print("\nTotal non-matching files:", len(non_matching_files))


# Valid Files

In [ ]:
# Valid files = files matching at least one approved filename pattern.
valid_files = state_df.loc[valid_mask].copy()

print("Total valid files:", len(valid_files))
print("Total invalid files:", len(state_df) - len(valid_files))


# Low  File Size(below 2000KB)

In [ ]:
state_df["FileSize"] = pd.to_numeric(
    state_df["FileSize"],
    errors="coerce"
)

lower_file_size = state_df.loc[
    state_df["FileSize"] <= 2000
].copy()

lower_file_size.to_excel(
    r"C:\Users\lenovo\Desktop\Survey Of India 2026 Files\July_Files\Rajasthan\Lower File Size.xlsx",
    index=False
)

print("Files <= 2000:", len(lower_file_size))
lower_file_size.head()


# Dupliactes Files

# Hourly File Data

In [ ]:
# Invalid files are the files that do NOT match any approved pattern.
invalid_files = state_df.loc[~valid_mask].copy()

invalid_file_list = invalid_files[
    ["Date", "FileName", "Station", "FileSize"]
].copy()

output_folder = r"C:\Users\lenovo\Desktop\Survey Of India 2026 Files\July_Files\Rajasthan"
os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(
    output_folder,
    "July_2026_Invalid_Files.xlsx"
)

invalid_file_list.to_excel(
    output_file,
    index=False
)

print("Total Invalid Files:", len(invalid_file_list))
print("File saved:", output_file)
invalid_file_list.head(20)


# Wrong Julian Code

In [ ]:
state_df["Date_Julian"] = state_df["Date"].dt.strftime("%j")

state_df["Path_Julian"] = state_df["FilePath"].astype(str).str.extract(
    r'[A-Za-z]{3,4}(\d{3})',
    expand=False
)

state_df["Julian_Match"] = (
    state_df["Date_Julian"] == state_df["Path_Julian"]
)

print(
    state_df[
        ["Date", "FileName", "Date_Julian", "Path_Julian", "Julian_Match"]
    ].head(20)
)


In [ ]:
julian_mismatch = state_df.loc[
    ~state_df["Julian_Match"]
].copy()

print("Julian mismatches:", len(julian_mismatch))
julian_mismatch


# Fetch the Daily File Distribution

In [ ]:
# Daily-wise distribution for all state files
daily_source = state_df.copy()

Daily_result = (
    daily_source
    .pivot_table(
        index="Date",
        columns="Station",
        aggfunc="size",
        fill_value=0
    )
    .sort_index()
)

print(Daily_result)


In [ ]:
# Save Daily Distribution to Excel

output_file = r"C:\Users\lenovo\Desktop\Survey Of India 2026 Files\July_Files\Rajasthan\Daily_Distribution.xlsx"

Daily_result.to_excel(
    output_file,
    index=True,
    sheet_name="Daily Distribution"
)

print("\nDaily distribution saved successfully:")
print(output_file)

# Daily Wise Distribution in Valid Files

In [ ]:
Daily_result_Valid_Files = (
    valid_files
    .pivot_table(
        index="Date",
        columns="Station",
        aggfunc="size",
        fill_value=0
    )
    .sort_index()
)

print(Daily_result_Valid_Files)


# Missing File

In [ ]:
# =========================================================
# MISSING FILE CALCULATION
# =========================================================

# IMPORTANT:
# This calculation assumes every selected station should produce
# 24 valid files for every day in July 2026.

expected_start = start_date
expected_end = end_date

expected_dates = pd.date_range(
    start=expected_start,
    end=expected_end,
    freq="D"
)

stations = sorted(
    state_df["Station"].dropna().astype(str).unique()
)

files_per_station_per_day = 24

valid_count = (
    valid_files
    .pivot_table(
        index="Date",
        columns="Station",
        aggfunc="size",
        fill_value=0
    )
    .reindex(
        index=expected_dates,
        columns=stations,
        fill_value=0
    )
)

missing_pivot = (
    files_per_station_per_day - valid_count
).clip(lower=0)

expected_files = (
    len(expected_dates)
    * len(stations)
    * files_per_station_per_day
)

actual_valid_files = len(valid_files)
total_missing_files = expected_files - actual_valid_files
pivot_missing_total = int(missing_pivot.sum().sum())

print("========================================")
print("JULY 2026 - MISSING FILE CALCULATION")
print("========================================")
print("Start Date           :", expected_start.strftime("%d-%m-%Y"))
print("End Date             :", expected_end.strftime("%d-%m-%Y"))
print("Number of Days       :", len(expected_dates))
print("Number of Stations   :", len(stations))
print("Files/Station/Day    :", files_per_station_per_day)
print("Expected Files       :", expected_files)
print("Actual Valid Files   :", actual_valid_files)
print("Missing Files        :", total_missing_files)
print("Pivot Missing Total  :", pivot_missing_total)

if total_missing_files == pivot_missing_total:
    print("✓ Counts match.")
else:
    print("⚠ Counts do not match.")

output_file = r"C:\Users\lenovo\Desktop\Survey Of India 2026 Files\July_Files\Rajasthan\Missing_File_Pivot.xlsx"

missing_pivot.to_excel(
    output_file,
    index=True,
    sheet_name="Missing Files"
)

print("Saved:", output_file)
print(missing_pivot)


In [ ]:
# Do not overwrite the missing-file report with the raw dataframe.
# The missing-file pivot is already saved in Cell 45.
print("Missing-file report was saved from missing_pivot in Cell 45.")


# Missing Character

In [ ]:
expected_chars = list("abcdefghijklmnopqrstuvwx")
missing_records = []

for (date, station), group in valid_files.groupby(["Date", "Station"]):
    filenames = group["FileName"].astype(str).tolist()

    actual_chars = {
        filename[7].lower()
        for filename in filenames
        if len(filename) > 7
    }

    missing_chars = [
        char for char in expected_chars
        if char not in actual_chars
    ]

    if not filenames:
        continue

    template_file = filenames[0]
    template_row = group.iloc[0]
    template_path = str(template_row["FilePath"])

    for missing_char in missing_chars:
        missing_file = (
            template_file[:7]
            + missing_char
            + template_file[8:]
        )

        expected_filepath = os.path.join(
            os.path.dirname(template_path),
            missing_file
        )

        missing_records.append({
            "Date": date,
            "Station": station,
            "Hourly_Missing_Character": missing_char,
            "Missing_File": missing_file,
            "Expected_FilePath": expected_filepath,
            "FileSize": 0
        })

missing_character_df = pd.DataFrame(missing_records)

output_file = r"C:\Users\lenovo\Desktop\Survey Of India 2026 Files\July_Files\Rajasthan\Missing_Character_File.xlsx"

missing_character_df.to_excel(
    output_file,
    index=False,
    sheet_name="Missing Hourly Files"
)

print("Total missing hourly files:", len(missing_character_df))
print("Saved:", output_file)
missing_character_df.head(20)


# Bar Chart

In [ ]:
state_df["FileSize"] = pd.to_numeric(
    state_df["FileSize"],
    errors="coerce"
)

bins = list(range(0, 12001, 1000))

size_range_count = (
    state_df.assign(
        Size_Range=pd.cut(
            state_df["FileSize"],
            bins=bins,
            right=True
        )
    )["Size_Range"]
    .value_counts()
    .sort_index()
    .reset_index()
)

size_range_count.columns = ["Size_Range", "File_Count"]

print("File count in each size range:")
print(size_range_count)


In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# Convert Size_Range to string
size_range_count["Size_Range"] = size_range_count["Size_Range"].astype(str)

# Normalize File_Count between 0 and 1 for color mapping
normalized = (size_range_count["File_Count"] - size_range_count["File_Count"].min()) / \
             (size_range_count["File_Count"].max() - size_range_count["File_Count"].min())

# Generate Viridis colors based on normalized File_Count
viridis_colors = px.colors.sample_colorscale("Viridis", normalized)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=size_range_count["Size_Range"],
    y=size_range_count["File_Count"],
    marker=dict(
        color=viridis_colors,       # Apply Viridis based on File_Count
        line=dict(color="black", width=1)
    ),
    text=[f"{int(count):,}" for count in size_range_count["File_Count"]],
    textposition='inside'
))

fig.update_layout(
    yaxis_type="log",
    xaxis_title="Size Range (KB)",
    yaxis_title="File Count (Log Scale)",
    title="File Count by Size Range (Filtered)",
    xaxis_tickangle=-45,
    width=1400,
    height=600,
    template="plotly_white"
)

fig

# Expected Output and Actual output File

In [ ]:
station_file_count = (
    valid_files
    .groupby("Station")
    .size()
    .reset_index(name="Output_file")
)

expected_per_station = len(expected_dates) * 24

station_file_count["Expected File"] = expected_per_station

station_file_count["Year"] = 2026

station_file_count.insert(
    0,
    "S.No",
    range(1, len(station_file_count) + 1)
)

station_file_count = station_file_count[
    ["S.No", "Year", "Station", "Output_file", "Expected File"]
]

print(station_file_count)

output_file = r"C:\Users\lenovo\Desktop\Survey Of India 2026 Files\July_Files\Rajasthan\Summary_Of_Station.xlsx"

station_file_count.to_excel(
    output_file,
    index=False,
    sheet_name="Station Summary"
)

print("\nStation-wise file summary saved successfully:")
print(output_file)
